# 연구 재시작 원본 보존: In[76] 정의 → In[77] 실행 → In[101] 결과

첨부 `디퓨전 2차 시험.ipynb`의 70–94번째 셀을 코드·출력 수정 없이 보존했습니다. 각 셀 앞에 원본 위치를 표시했습니다. 기본 MLP, 구조 encoder MLP, pathway 없는 6조건(v-diffusion), 평균 pathway MLP 4조건, 평균 pathway 12조건(epsilon-DDPM)을 포함합니다. sparse 9조건은 별도입니다.

**원본 보존용이며 Run All하면 학습·파일 쓰기가 실행됩니다. 추가 학습은 아직 승인되지 않았습니다.** 현재 서버 파일과 기존 참조 결과가 필요하며, 원본 83번째의 snapshot 저장 셀은 `columns`, `audit` 등 생성 정의가 빠져 있어 이 파일만으로 실행할 수 없습니다. 독립 실행 완료본이라고 주장하지 않습니다.

매핑부터 새로 준비하려면 `docs/mapping-and-restart.md`와 재구성 코드 `scripts/reconstruct_mapping.py`를 참고하세요. 이는 원본과 별도이며 기존 실험 입력의 생성 이력을 복원하지 않습니다.


## 원본 70번째 셀 · In[76]


In [76]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in [('expression_only',False),('expression_structure',True)]:
        model=model_for(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_for(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



## 원본 71번째 셀 · In[77]


In [77]:
from argparse import Namespace
from datetime import datetime
import uuid

DATA_ROOT = Path('/home/seowon1211/data1')
EPOCHS = 30
FIRST_OUT = DATA_ROOT/'conditioned_expression'/(
    'first_structure_mlp_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '_' + uuid.uuid4().hex[:6]
)
a_first = Namespace(
    prepared=DATA_ROOT/'conditioned_expression/prepared_controls',
    structure=DATA_ROOT/'conditioned_expression/morgan_matched/morgan_binary.csv',
    out=FIRST_OUT,epochs=EPOCHS,check_every=5,batch_size=128,threads=4,seed=42,
)
assert a_first.prepared.is_dir(),a_first.prepared
assert a_first.structure.is_file(),a_first.structure
print('구조 유무 최소 비교 시작:',FIRST_OUT,flush=True)
with threadpool_limits(limits=a_first.threads):
    first_comparison=run(a_first)
print('실험 완료',flush=True)


구조 유무 최소 비교 시작: /home/seowon1211/data1/conditioned_expression/first_structure_mlp_20261102_023222_1baa95
Compounds train/val/test: [89, 23, 28]; probes=3453
Missing structure compounds: ['LPS', 'TNFÉø', 'tunicamycin']
expression_only epoch=1/30, val_RMSE=0.30233, best_epoch=1
expression_only epoch=5/30, val_RMSE=0.28567, best_epoch=5
expression_only epoch=10/30, val_RMSE=0.27646, best_epoch=10
expression_only epoch=15/30, val_RMSE=0.27742, best_epoch=10
expression_only epoch=20/30, val_RMSE=0.27647, best_epoch=10
expression_only epoch=25/30, val_RMSE=0.27717, best_epoch=10
expression_only epoch=30/30, val_RMSE=0.27879, best_epoch=10
expression_structure epoch=1/30, val_RMSE=0.30386, best_epoch=1
expression_structure epoch=5/30, val_RMSE=0.28554, best_epoch=5
expression_structure epoch=10/30, val_RMSE=0.27968, best_epoch=10
expression_structure epoch=15/30, val_RMSE=0.28043, best_epoch=10
expression_structure epoch=20/30, val_RMSE=0.27851, best_epoch=20
expression_structure epoch=25/30,

## 원본 72번째 셀 · In[78]


In [78]:
print('첫 미팅용 비교표')
display(first_comparison[['model','absolute_rmse','response_rmse',
    'zero_response_rmse','ratio_to_no_change','pair_rmse','selected_epoch','parameters']])

base=first_comparison.set_index('model').loc['expression_only']
added=first_comparison.set_index('model').loc['expression_structure']
print('구조 추가의 절대 발현 RMSE 변화:',added.absolute_rmse-base.absolute_rmse)
print('구조 추가의 반응 RMSE 변화:',added.response_rmse-base.response_rmse)
print('음수면 구조 추가 모델이 더 작습니다.')
print('결과 파일:',FIRST_OUT/'comparison.csv')


첫 미팅용 비교표


,model,absolute_rmse,response_rmse,zero_response_rmse,ratio_to_no_change,pair_rmse,selected_epoch,parameters
0,expression_only,0.323074,0.315112,0.330116,0.954549,0.323272,10,939261
1,expression_structure,0.325483,0.317698,0.330116,0.962381,0.323806,20,939261


구조 추가의 절대 발현 RMSE 변화: 0.002409320176539387
구조 추가의 반응 RMSE 변화: 0.002585614551896709
음수면 구조 추가 모델이 더 작습니다.
결과 파일: /home/seowon1211/data1/conditioned_expression/first_structure_mlp_20261102_023222_1baa95/comparison.csv


## 원본 73번째 셀 · In[79]


In [79]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a,model_factory=None):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    model_factory=model_for if model_factory is None else model_factory
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in [('expression_only',False),('expression_structure',True)]:
        model=model_factory(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_factory(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



## 원본 74번째 셀 · In[80]


In [80]:
"""Minimal follow-up: separate expression/structure encoders with a matched control."""
import argparse
from pathlib import Path
import torch
from torch import nn
from threadpoolctl import threadpool_limits
from types import SimpleNamespace
first = SimpleNamespace(run=run,seed_all=seed_all)


class SeparateEncoderMLP(nn.Module):
    def __init__(self,source_dim,condition_dim,structure_dim,output_dim):
        super().__init__()
        self.expression_encoder=nn.Sequential(
            nn.Linear(source_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU())
        # No biases: a zero structure (including its presence flag) maps to exact zero.
        self.structure_encoder=nn.Sequential(
            nn.Linear(structure_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),
            nn.Linear(64,32,bias=False),nn.SiLU())
        self.decoder=nn.Sequential(
            nn.Linear(64+32+condition_dim,64),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,output_dim))

    def forward(self,source,condition,structure):
        expression=self.expression_encoder(source)
        molecular=self.structure_encoder(structure)
        return self.decoder(torch.cat([expression,molecular,condition],dim=1))


def model_for(data,pack,seed):
    first.seed_all(seed)
    return SeparateEncoderMLP(pack['source'].shape[1],pack['conditions'].shape[1],pack['structure'].shape[1],data.target.shape[1])


def run_encoder(a):
    a.architecture='expression_128_64__structure_64_32__fusion_64__full_probe_output'
    a.structure_control='identical architecture; structure and presence inputs zeroed; bias-free structure encoder'
    return first.run(a,model_factory=model_for)



## 원본 75번째 셀 · In[81]


In [81]:
from argparse import Namespace
from datetime import datetime
import uuid

DATA_ROOT = Path('/home/seowon1211/data1')
EPOCHS = 30
ENCODER_OUT = DATA_ROOT/'conditioned_expression'/(
    'structure_encoder_mlp_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '_' + uuid.uuid4().hex[:6]
)
a_encoder = Namespace(
    prepared=DATA_ROOT/'conditioned_expression/prepared_controls',
    structure=DATA_ROOT/'conditioned_expression/morgan_matched/morgan_binary.csv',
    out=ENCODER_OUT,epochs=EPOCHS,check_every=5,batch_size=128,threads=4,seed=42,
)
assert a_encoder.prepared.is_dir(),a_encoder.prepared
assert a_encoder.structure.is_file(),a_encoder.structure
print('별도 구조 encoder 비교 시작:',ENCODER_OUT,flush=True)
with threadpool_limits(limits=a_encoder.threads):
    encoder_comparison=run_encoder(a_encoder)
print('비교 완료',flush=True)


별도 구조 encoder 비교 시작: /home/seowon1211/data1/conditioned_expression/structure_encoder_mlp_20261102_025712_f0455c
Compounds train/val/test: [89, 23, 28]; probes=3453
Missing structure compounds: ['LPS', 'TNFÉø', 'tunicamycin']
expression_only epoch=1/30, val_RMSE=0.29270, best_epoch=1
expression_only epoch=5/30, val_RMSE=0.29008, best_epoch=5
expression_only epoch=10/30, val_RMSE=0.28483, best_epoch=10
expression_only epoch=15/30, val_RMSE=0.28685, best_epoch=10
expression_only epoch=20/30, val_RMSE=0.29716, best_epoch=10
expression_only epoch=25/30, val_RMSE=0.29743, best_epoch=10
expression_only epoch=30/30, val_RMSE=0.30571, best_epoch=10
expression_structure epoch=1/30, val_RMSE=0.29389, best_epoch=1
expression_structure epoch=5/30, val_RMSE=0.29334, best_epoch=5
expression_structure epoch=10/30, val_RMSE=0.28979, best_epoch=10
expression_structure epoch=15/30, val_RMSE=0.29210, best_epoch=10
expression_structure epoch=20/30, val_RMSE=0.30203, best_epoch=10
expression_structure epoch

## 원본 76번째 셀 · In[82]


In [82]:
print('새 architecture 안에서 구조 입력 유무 비교')
display(encoder_comparison[['model','absolute_rmse','response_rmse',
    'zero_response_rmse','ratio_to_no_change','pair_rmse','selected_epoch','parameters']])
base_row=encoder_comparison.set_index('model').loc['expression_only']
structure_row=encoder_comparison.set_index('model').loc['expression_structure']
print('구조 추가의 절대 발현 RMSE 변화:',structure_row.absolute_rmse-base_row.absolute_rmse)
print('구조 추가의 반응 RMSE 변화:',structure_row.response_rmse-base_row.response_rmse)
print('음수면 구조 추가가 더 좋습니다.')

previous=DATA_ROOT/'conditioned_expression/first_structure_mlp_20261102_023222_1baa95/compound_split.csv'
if previous.is_file():
    old=pd.read_csv(previous).sort_values('compound').reset_index(drop=True)
    current=pd.read_csv(ENCODER_OUT/'compound_split.csv').sort_values('compound').reset_index(drop=True)
    pd.testing.assert_frame_equal(old,current)
    print('첫 실험과 같은 화합물 분할 확인 완료')
print('결과:',ENCODER_OUT/'comparison.csv')


새 architecture 안에서 구조 입력 유무 비교


,model,absolute_rmse,response_rmse,zero_response_rmse,ratio_to_no_change,pair_rmse,selected_epoch,parameters
0,expression_only,0.322968,0.314199,0.330116,0.951783,0.321989,10,815293
1,expression_structure,0.335091,0.310973,0.330116,0.942010,0.330712,10,815293


구조 추가의 절대 발현 RMSE 변화: 0.012122459349772796
구조 추가의 반응 RMSE 변화: -0.0032261629994281282
음수면 구조 추가가 더 좋습니다.
첫 실험과 같은 화합물 분할 확인 완료
결과: /home/seowon1211/data1/conditioned_expression/structure_encoder_mlp_20261102_025712_f0455c/comparison.csv


## 원본 77번째 셀 · In[83]


In [83]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a,model_factory=None):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    model_factory=model_for if model_factory is None else model_factory
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in [('expression_only',False),('expression_structure',True)]:
        model=model_factory(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_factory(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



## 원본 78번째 셀 · In[84]


In [84]:
class SeparateEncoderMLP(nn.Module):
    def __init__(self,source_dim,condition_dim,structure_dim,output_dim):
        super().__init__()
        self.expression_encoder=nn.Sequential(
            nn.Linear(source_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU())
        # No biases: a zero structure (including its presence flag) maps to exact zero.
        self.structure_encoder=nn.Sequential(
            nn.Linear(structure_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),
            nn.Linear(64,32,bias=False),nn.SiLU())
        self.decoder=nn.Sequential(
            nn.Linear(64+32+condition_dim,64),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,output_dim))

    def forward(self,source,condition,structure):
        expression=self.expression_encoder(source)
        molecular=self.structure_encoder(structure)
        return self.decoder(torch.cat([expression,molecular,condition],dim=1))

"""Minimal meeting comparison: cGAN/v-DDPM, own structure on/off, existing MLP reference."""
import argparse
import copy
import json
from pathlib import Path
from types import SimpleNamespace
import uuid
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from threadpoolctl import threadpool_limits
first = SimpleNamespace(Data=Data,prepare=prepare,seed_all=seed_all,inputs=inputs,evaluate=evaluate)



class Context(nn.Module):
    def __init__(self,source_dim,condition_dim,structure_dim,output_dim):
        super().__init__()
        template=SeparateEncoderMLP(source_dim,condition_dim,structure_dim,output_dim)
        self.expression_encoder=template.expression_encoder
        self.structure_encoder=template.structure_encoder
        self.fusion=template.decoder[:3]
    def forward(self,source,condition,structure):
        return self.fusion(torch.cat([self.expression_encoder(source),self.structure_encoder(structure),condition],1))


class Generator(nn.Module):
    def __init__(self,dims):
        super().__init__();self.context=Context(*dims);self.noise_dim=16
        self.head=nn.Sequential(nn.Linear(64+16,64),nn.SiLU(),nn.Linear(64,dims[-1]))
    def decode(self,context,noise):return self.head(torch.cat([context,noise],1))
    def forward(self,source,condition,structure,noise):return self.decode(self.context(source,condition,structure),noise)


class Discriminator(nn.Module):
    def __init__(self,dims):
        super().__init__();self.context=Context(*dims)
        self.head=nn.Sequential(nn.Linear(64+dims[-1],128),nn.LeakyReLU(.2),nn.Linear(128,64),nn.LeakyReLU(.2),nn.Linear(64,1))
    def forward(self,source,condition,structure,expression):
        return self.head(torch.cat([self.context(source,condition,structure),expression],1)).squeeze(1)


class Denoiser(nn.Module):
    def __init__(self,dims,steps):
        super().__init__();self.context=Context(*dims);self.time=nn.Embedding(steps,16);self.output_dim=dims[-1]
        self.head=nn.Sequential(nn.Linear(64+dims[-1]+16,128),nn.SiLU(),nn.Linear(128,128),nn.SiLU(),nn.Linear(128,dims[-1]))
    def decode(self,context,noisy,t):return self.head(torch.cat([context,noisy,self.time(t)],1))
    def forward(self,source,condition,structure,noisy,t):return self.decode(self.context(source,condition,structure),noisy,t)


def schedule(steps):
    t=torch.linspace(0,steps,steps+1)
    cumulative=torch.cos(((t/steps+.008)/1.008)*torch.pi/2).square();cumulative/=cumulative[0].clone()
    beta=(1-cumulative[1:]/cumulative[:-1]).clamp(.0001,.999)
    return torch.cumprod(1-beta,0)


def v_to_x0_epsilon(noisy,v,abar):
    return abar.sqrt()*noisy-(1-abar).sqrt()*v,(1-abar).sqrt()*noisy+abar.sqrt()*v


def make_models(kind,data,pack,a):
    first.seed_all(a.seed)
    dims=(pack['source'].shape[1],pack['conditions'].shape[1],pack['structure'].shape[1],data.target.shape[1])
    model=Generator(dims) if kind=='gan' else Denoiser(dims,a.diffusion_steps)
    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
    disc=Discriminator(dims) if kind=='gan' else None
    dopt=torch.optim.AdamW(disc.parameters(),lr=.001,weight_decay=.01) if disc is not None else None
    return model,opt,disc,dopt


def train_epoch(kind,model,opt,disc,dopt,data,pack,tr,structure,a):
    model.train()
    if disc is not None:disc.train()
    total=0.;abar=schedule(a.diffusion_steps)
    for local in torch.randperm(len(tr)).split(a.batch_size):
        rows=tr[local.numpy()];x=first.inputs(data,pack,rows,structure)
        y=pack['target'][data.target_ids[rows]];weights=pack['weights'][local]
        if kind=='gan':
            for p in disc.parameters():p.requires_grad_(True)
            fake=model(*x,torch.randn(len(rows),16)).detach()
            real_logits=disc(*x,y);fake_logits=disc(*x,fake)
            dloss=.5*((F.binary_cross_entropy_with_logits(real_logits,torch.ones_like(real_logits),reduction='none')+F.binary_cross_entropy_with_logits(fake_logits,torch.zeros_like(fake_logits),reduction='none'))*weights).mean()
            if not torch.isfinite(dloss):raise ValueError('Nonfinite discriminator loss')
            dopt.zero_grad();dloss.backward();nn.utils.clip_grad_norm_(disc.parameters(),1.);dopt.step()
            for p in disc.parameters():p.requires_grad_(False)
            pred=model(*x,torch.randn(len(rows),16));logits=disc(*x,pred)
            reconstruction=(pred-y).square().mean(1)
            adversarial=F.binary_cross_entropy_with_logits(logits,torch.ones_like(logits),reduction='none')
            loss=((reconstruction+a.adversarial_weight*adversarial)*weights).mean()
        else:
            t=torch.randint(a.diffusion_steps,(len(rows),));noise=torch.randn_like(y)
            noisy=abar[t,None].sqrt()*y+(1-abar[t,None]).sqrt()*noise
            v=abar[t,None].sqrt()*noise-(1-abar[t,None]).sqrt()*y
            loss=((model(*x,noisy,t)-v).square().mean(1)*weights).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step();total+=float(loss.detach())*len(rows)
    return total/len(tr)


@torch.no_grad()
def draw_mean(kind,model,inputs,a,seed):
    model.eval();gen=torch.Generator().manual_seed(seed)
    context=model.context(*inputs);n=len(context)
    context=context.repeat_interleave(a.samples,0)
    if kind=='gan':pred=model.decode(context,torch.randn((len(context),16),generator=gen))
    else:
        abar=schedule(a.diffusion_steps)
        pred=torch.randn((len(context),model.output_dim),generator=gen)
        times=np.linspace(a.diffusion_steps-1,0,a.sampling_steps,dtype=int)
        # Deterministic DDIM updates; independent initial noise gives independent samples.
        for i,step in enumerate(times):
            v=model.decode(context,pred,torch.full((len(context),),int(step),dtype=torch.long))
            x0,eps=v_to_x0_epsilon(pred,v,abar[step])
            previous=abar[times[i+1]] if i+1<len(times) else torch.tensor(1.)
            pred=previous.sqrt()*x0+(1-previous).sqrt()*eps
    if not torch.isfinite(pred).all():raise ValueError('Nonfinite generated values')
    return pred.reshape(n,a.samples,-1).mean(1).numpy()


@torch.no_grad()
def predict(kind,model,data,pack,rows,structure,a,seed):
    ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),float);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),a.batch_size):
        batch=rows[start:start+a.batch_size]
        pred=draw_mean(kind,model,first.inputs(data,pack,batch,structure),a,seed+start)+pack['target_mean']
        target=data.target_ids[batch];indices=np.array([lookup[int(cid)] for cid in target])
        np.add.at(sums,indices,pred);np.add.at(counts,indices,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def save_progress(path,manifest):
    temporary=path.with_suffix('.tmp');temporary.write_text(json.dumps(manifest,indent=2));temporary.replace(path)


def run_comparison(a):
    reference=Path(a.mlp_reference)
    old_cfg=json.loads((reference/'config.json').read_text())
    for field in ['prepared','structure']:
        if Path(old_cfg[field]).resolve()!=Path(getattr(a,field)).resolve():raise ValueError(f'MLP reference {field} mismatch')
    for field in ['seed','epochs','check_every','batch_size']:
        if old_cfg[field]!=getattr(a,field):raise ValueError(f'MLP reference differs in {field}; use matched settings')
    torch.set_num_threads(a.threads);data=first.Data(a)
    split=pd.read_csv(reference/'compound_split.csv')
    if split.compound.duplicated().any() or set(split.compound)!=set(data.names):raise ValueError('Reference split mismatch')
    def rows(stage):return np.flatnonzero(data.pairs.compound.isin(set(split.loc[split.stage.eq(stage),'compound'])))
    itr,iv,te=rows('inner_train'),rows('inner_validation'),rows('outer_test')
    tr=np.sort(np.r_[itr,iv])
    if any(set(x)&set(y) for x,y in [(itr,iv),(itr,te),(iv,te)]):raise ValueError('Split overlap')
    reference_rows=pd.read_csv(reference/'comparison.csv')
    if set(reference_rows.model)!={'expression_only','expression_structure'}:raise ValueError('Both MLP variants are required')
    for name in ['expression_only','expression_structure']:
        with np.load(reference/f'{name}_predictions.npz') as prediction:
            ids,pred=prediction['target_ids'],prediction['prediction']
        np.testing.assert_array_equal(ids,np.unique(data.target_ids[te]))
        verified,_=first.evaluate(data,ids,pred)
        old=reference_rows.set_index('model').loc[name]
        for key in ['absolute_rmse','response_rmse']:np.testing.assert_allclose(verified[key],old[key],rtol=1e-5,atol=1e-6)
    a.out.mkdir(parents=True,exist_ok=True)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same saved MLP split and preprocessing; same encoder architecture; all probes directly',diffusion='v-DDPM training and deterministic DDIM sampling, no target PCA or x0 clipping',gan='conditional reconstruction MSE + adversarial BCE; not full AIVIVE reproduction')
    config_path=a.out/'config.json'
    if config_path.exists():
        if json.loads(config_path.read_text())!=config:raise ValueError('Resume settings mismatch')
    else:
        if any(a.out.iterdir()):raise ValueError('Unrecognized output directory')
        config_path.write_text(json.dumps(config,indent=2))
    split.to_csv(a.out/'compound_split.csv',index=False);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    progress_path=a.out/'progress.json'
    manifest=json.loads(progress_path.read_text()) if progress_path.is_file() else {'runs':{}}
    result=reference_rows.copy();result['family']='MLP';result['structure']=result.model.eq('expression_structure');result['origin']='reused_MLP_reference';result['discriminator_parameters']=0
    completed=result.to_dict('records')
    print('MLP results verified and reused. Four new runs: GAN/diffusion x structure off/on.',flush=True)
    inner=first.prepare(data,itr,iv);outer=first.prepare(data,tr,te)
    saved_pre=joblib.load(reference/'preprocess.joblib')
    for attr in ['mean_','scale_']:np.testing.assert_allclose(getattr(outer['preprocess']['source_scale'],attr),getattr(saved_pre['source_scale'],attr))
    np.testing.assert_allclose(outer['target_mean'],saved_pre['target_mean'])
    joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    for kind in ['gan','diffusion']:
        for structure in [False,True]:
            name=f'{kind}_'+('expression_structure' if structure else 'expression_only')
            entry=manifest['runs'].get(name)
            if entry and (Path(entry['path'])/'metrics.json').is_file():
                metrics=json.loads((Path(entry['path'])/'metrics.json').read_text());completed.append(metrics)
                print(name,': completed result reused',flush=True);continue
            if entry:print(name,': partial attempt preserved; restarting this variant',flush=True)
            folder=a.out/f'{name}_{uuid.uuid4().hex[:6]}';folder.mkdir()
            manifest['runs'][name]={'path':str(folder),'status':'running'};save_progress(progress_path,manifest)
            print(f'\n{name}: inner training starts; output={folder}',flush=True)
            model,opt,disc,dopt=make_models(kind,data,inner,a)
            best=float('inf');best_epoch=None;state=None;history=[]
            for epoch in range(1,a.epochs+1):
                loss=train_epoch(kind,model,opt,disc,dopt,data,inner,itr,structure,a)
                if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                    ids,pred,_=predict(kind,model,data,inner,iv,structure,a,a.seed+20000)
                    mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                    if mse<best:best=mse;best_epoch=epoch;state=copy.deepcopy(model.state_dict())
                    history.append(dict(epoch=epoch,training_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                    pd.DataFrame(history).to_csv(folder/'validation_history.csv',index=False)
                    print(f'{name} epoch={epoch}/{a.epochs} val_RMSE={np.sqrt(mse):.5f} best_epoch={best_epoch}',flush=True)
            torch.save(state,folder/'inner_best.pt')
            model,opt,disc,dopt=make_models(kind,data,outer,a)
            for epoch in range(1,best_epoch+1):
                train_epoch(kind,model,opt,disc,dopt,data,outer,tr,structure,a)
                if epoch==1 or epoch%10==0 or epoch==best_epoch:print(f'{name} refit={epoch}/{best_epoch}',flush=True)
            ids,pred,pair_rmse=predict(kind,model,data,outer,te,structure,a,a.seed+30000)
            metrics,detail=first.evaluate(data,ids,pred)
            metrics.update(model=name,family='cGAN' if kind=='gan' else 'v-diffusion',structure=structure,origin='new_run',selected_epoch=best_epoch,validation_rmse=float(np.sqrt(best)),pair_rmse=pair_rmse,
                parameters=sum(p.numel() for p in model.parameters()),discriminator_parameters=sum(p.numel() for p in disc.parameters()) if disc is not None else 0)
            np.savez_compressed(folder/'predictions.npz',target_ids=ids,prediction=pred)
            torch.save(model.state_dict(),folder/'model.pt')
            if disc is not None:torch.save(disc.state_dict(),folder/'discriminator.pt')
            detail.to_csv(folder/'response_condition_metrics.csv',index=False)
            (folder/'metrics.json').write_text(json.dumps(metrics,indent=2))
            manifest['runs'][name]['status']='complete';save_progress(progress_path,manifest)
            completed.append(metrics);pd.DataFrame(completed).to_csv(a.out/'comparison.csv',index=False)
            print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    result=pd.DataFrame(completed);result.to_csv(a.out/'comparison.csv',index=False)
    print(result.to_string(index=False));print('Saved:',a.out,flush=True)
    return result



## 원본 79번째 셀 · In[85]


In [85]:
from datetime import datetime
import uuid

DATA_ROOT = Path('/home/seowon1211/data1')
MLP_REFERENCE = DATA_ROOT/'conditioned_expression/structure_encoder_mlp_20261102_025712_f0455c'
assert (MLP_REFERENCE/'comparison.csv').is_file(),MLP_REFERENCE
reference_config=json.loads((MLP_REFERENCE/'config.json').read_text())

# 새 커널에서 중단된 실행을 재개할 때만 이전 출력 폴더 입력
RESUME_ROOT = None
if RESUME_ROOT is None:
    GENERATIVE_OUT = DATA_ROOT/'conditioned_expression'/(
        'meeting_gan_diffusion_' + datetime.now().strftime('%Y%m%d_%H%M%S')
        + '_' + uuid.uuid4().hex[:6]
    )
else:
    GENERATIVE_OUT = Path(RESUME_ROOT)

a_generative = SimpleNamespace(
    prepared=Path(reference_config['prepared']),structure=Path(reference_config['structure']),
    mlp_reference=MLP_REFERENCE,out=GENERATIVE_OUT,
    epochs=reference_config['epochs'],check_every=reference_config['check_every'],
    batch_size=reference_config['batch_size'],threads=4,seed=reference_config['seed'],
    samples=4,diffusion_steps=30,sampling_steps=10,adversarial_weight=.1,
)
print('기존 MLP:',MLP_REFERENCE,flush=True)
print('새 결과/재개 폴더:',GENERATIVE_OUT,flush=True)


기존 MLP: /home/seowon1211/data1/conditioned_expression/structure_encoder_mlp_20261102_025712_f0455c
새 결과/재개 폴더: /home/seowon1211/data1/conditioned_expression/meeting_gan_diffusion_20261102_031308_af1f40


## 원본 80번째 셀 · In[86]


In [86]:
print('GAN/diffusion 네 실험 시작 — 기존 MLP는 재사용합니다.',flush=True)
with threadpool_limits(limits=a_generative.threads):
    meeting_comparison = run_comparison(a_generative)
print('전체 비교 완료',flush=True)


GAN/diffusion 네 실험 시작 — 기존 MLP는 재사용합니다.
MLP results verified and reused. Four new runs: GAN/diffusion x structure off/on.

gan_expression_only: inner training starts; output=/home/seowon1211/data1/conditioned_expression/meeting_gan_diffusion_20261102_031308_af1f40/gan_expression_only_399017
gan_expression_only epoch=1/30 val_RMSE=0.36168 best_epoch=1
gan_expression_only epoch=5/30 val_RMSE=0.52563 best_epoch=1
gan_expression_only epoch=10/30 val_RMSE=0.33526 best_epoch=10
gan_expression_only epoch=15/30 val_RMSE=0.31829 best_epoch=15
gan_expression_only epoch=20/30 val_RMSE=0.33528 best_epoch=15
gan_expression_only epoch=25/30 val_RMSE=0.32326 best_epoch=15
gan_expression_only epoch=30/30 val_RMSE=0.33028 best_epoch=15
gan_expression_only refit=1/15
gan_expression_only refit=10/15
gan_expression_only refit=15/15
gan_expression_only: TEST absolute=0.37978, response=0.33406

gan_expression_structure: inner training starts; output=/home/seowon1211/data1/conditioned_expression/meeting_gan_

## 원본 81번째 셀 · In[87]


In [87]:
print('첫 미팅용 6행 비교표')
display(meeting_comparison[['family','structure','absolute_rmse','response_rmse',
    'zero_response_rmse','ratio_to_no_change','pair_rmse','selected_epoch',
    'parameters','discriminator_parameters']])

structure_effect=meeting_comparison.pivot(index='family',columns='structure',values=['absolute_rmse','response_rmse'])
print('각 모델에서 구조 추가의 변화: 음수면 구조 추가가 더 좋습니다.')
rows=[]
for family,part in meeting_comparison.groupby('family'):
    base=part.loc[~part.structure].iloc[0]
    augmented=part.loc[part.structure].iloc[0]
    rows.append(dict(model=family,absolute_RMSE_change=augmented.absolute_rmse-base.absolute_rmse,
        response_RMSE_change=augmented.response_rmse-base.response_rmse))
structure_effect_table=pd.DataFrame(rows)
display(structure_effect_table)
structure_effect_table.to_csv(GENERATIVE_OUT/'structure_effect.csv',index=False)
print('결과:',GENERATIVE_OUT/'comparison.csv')


첫 미팅용 6행 비교표


,family,structure,absolute_rmse,response_rmse,zero_response_rmse,ratio_to_no_change,pair_rmse,selected_epoch,parameters,discriminator_parameters
0,MLP,False,0.322968,0.314199,0.330116,0.951783,0.321989,10,815293,0
1,MLP,True,0.335091,0.310973,0.330116,0.942010,0.330712,10,815293,0
2,cGAN,False,0.379776,0.334065,0.330116,1.011960,0.391336,15,820477,1049473
3,cGAN,True,0.368399,0.332743,0.330116,1.007958,0.372057,25,820477,1049473
4,v-diffusion,False,0.353868,0.375389,0.330116,1.137143,0.545726,30,1505629,0
5,v-diffusion,True,0.373495,0.374674,0.330116,1.134976,0.557739,10,1505629,0


각 모델에서 구조 추가의 변화: 음수면 구조 추가가 더 좋습니다.


,model,absolute_RMSE_change,response_RMSE_change
0,MLP,0.012122,-0.003226
1,cGAN,-0.011377,-0.001321
2,v-diffusion,0.019628,-0.000715


결과: /home/seowon1211/data1/conditioned_expression/meeting_gan_diffusion_20261102_031308_af1f40/comparison.csv


## 원본 82번째 셀 · In[88]


In [88]:
from pathlib import Path
import os
import pandas as pd

DATA = Path("/home/seowon1211/data1")

genes = pd.read_csv(DATA / "AIVIVE/data/final_rat_genes.csv")
print("유전자 매핑 컬럼:", genes.columns.tolist())
display(genes.head())

keywords = ("pathway", "kegg", "reactome", "gene2go", "gene_sets")
skip = {
    ".git", ".cache", ".ipynb_checkpoints", "__pycache__",
    "node_modules", "site-packages", "conditioned_expression",
}
found = []

for directory, dirs, files in os.walk(DATA):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".") and d not in skip
    ]
    for filename in files:
        path = Path(directory) / filename
        name = filename.lower()
        if (
            path.suffix.lower() == ".gmt"
            or (
                any(word in name for word in keywords)
                and path.suffix.lower() in {".csv", ".tsv", ".txt"}
            )
        ):
            found.append(path)

print("\npathway 관련 후보 파일:")
for path in found[:40]:
    print(path)

if not found:
    print("후보 파일 없음 — 외부의 고정된 Rat 유전자 집합이 필요합니다.")

print("\nAIVIVE 공발현 모듈 파일 — pathway와 별도로 취급:")
module_dir = DATA / "AIVIVE/data/modules_genes"
if module_dir.is_dir():
    for path in sorted(module_dir.iterdir())[:5]:
        print(path)

유전자 매핑 컬럼: ['PROBEID', 'GENENAME', 'SYMBOL']


,PROBEID,GENENAME,SYMBOL
0,1367454_at,COPI coat complex subunit beta 2,Copb2
1,1367457_at,beclin 1,Becn1
2,1367459_at,ADP-ribosylation factor 1,Arf1
3,1367464_at,poly-U binding splicing factor 60,Puf60
4,1367468_at,SCAN domain-containing 1,Scand1



pathway 관련 후보 파일:
후보 파일 없음 — 외부의 고정된 Rat 유전자 집합이 필요합니다.

AIVIVE 공발현 모듈 파일 — pathway와 별도로 취급:
/home/seowon1211/data1/AIVIVE/data/modules_genes/module115_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module120_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module15_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module176_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module18_genes.csv


## 원본 83번째 셀 · In[90]


In [90]:
# 계산·검증은 float64로 수행
projection = np.column_stack(columns).astype(np.float64)
column_sums = projection.sum(axis=0)

assert np.isfinite(projection).all()
assert (projection >= 0).all()
assert (column_sums > 0).all()

# 정규화 전에 차이가 실제로 반올림 수준인지 확인
np.testing.assert_allclose(
    column_sums, 1.0,
    rtol=0, atol=1e-6,
)

projection = projection / column_sums[None, :]
np.testing.assert_allclose(
    projection.sum(axis=0), 1.0,
    rtol=0, atol=1e-12,
)

audit.to_csv(PATHWAY_OUT / "probe_mapping_audit.csv", index=False)

pathway_table = pd.DataFrame(pathway_audit)
pathway_table.to_csv(PATHWAY_OUT / "pathway_audit.csv", index=False)

np.save(PATHWAY_OUT / "projection.npy", projection)

(PATHWAY_OUT / "provenance.json").write_text(json.dumps({
    "species": "Rattus norvegicus",
    "sources": provenance,
    "minimum_matched_genes": 5,
    "maximum_total_pathway_genes": 500,
    "probe_order": probe_order.tolist(),
    "projection_dtype": "float64",
}, ensure_ascii=False, indent=2))

print("전체 probe:", len(audit))
print("KEGG 유전자에 연결된 probe:", int(audit.kegg_gene.notna().sum()))
print("사용 가능한 pathway:", len(pathway_table))
print("저장 위치:", PATHWAY_OUT)
display(pathway_table.head(10))

전체 probe: 3453
KEGG 유전자에 연결된 probe: 3410
사용 가능한 pathway: 321
저장 위치: /home/seowon1211/data1/conditioned_expression/kegg_rat_20261102_034010_265aed


,pathway,name,total_kegg_genes,matched_genes,matched_probes
0,rno00010,Glycolysis / Gluconeogenesis - Rattus norvegic...,87,25,37
1,rno00020,Citrate cycle (TCA cycle) - Rattus norvegicus ...,32,11,13
2,rno00030,Pentose phosphate pathway - Rattus norvegicus ...,32,14,17
3,rno00040,Pentose and glucuronate interconversions - Rat...,37,8,11
4,rno00051,Fructose and mannose metabolism - Rattus norve...,39,11,18
5,rno00052,Galactose metabolism - Rattus norvegicus (rat),33,12,25
6,rno00053,Ascorbate and aldarate metabolism - Rattus nor...,32,9,11
7,rno00062,Fatty acid elongation - Rattus norvegicus (rat),31,12,16
8,rno00071,Fatty acid degradation - Rattus norvegicus (rat),50,23,32
9,rno00100,Steroid biosynthesis - Rattus norvegicus (rat),20,12,20


## 원본 84번째 셀 · In[91]


In [91]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a,model_factory=None,prepare_fn=None):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    model_factory=model_for if model_factory is None else model_factory
    prepare_fn=prepare if prepare_fn is None else prepare_fn
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare_fn(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in [('expression_only',False),('expression_structure',True)]:
        model=model_factory(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare_fn(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_factory(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



## 원본 85번째 셀 · In[92]


In [92]:
"""Four-way MLP ablation: expression, own structure, pathway summary, both."""
import argparse
import hashlib
import json
from pathlib import Path
from types import SimpleNamespace
import uuid
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
first = SimpleNamespace(run=run,prepare=prepare,seed_all=seed_all)


def load_projection(annotation,prepared):
    annotation=Path(annotation)
    provenance=json.loads((annotation/'provenance.json').read_text())
    probes=pd.read_csv(Path(prepared)/'probes.csv')['probe'].astype(str).tolist()
    if provenance['species']!='Rattus norvegicus' or provenance['probe_order']!=probes:raise ValueError('Species/probe order mismatch')
    projection=np.load(annotation/'projection.npy').astype(np.float64)
    pathways=pd.read_csv(annotation/'pathway_audit.csv')
    if projection.shape!=(len(probes),len(pathways)) or len(pathways)==0 or pathways.pathway.duplicated().any():raise ValueError('Pathway layout mismatch')
    if not np.isfinite(projection).all() or (projection<0).any():raise ValueError('Invalid pathway weights')
    np.testing.assert_allclose(projection.sum(0),1.,rtol=0,atol=1e-6)
    return projection,pathways,provenance


def prepare_pathways(data,tr,ev,projection,pathways,provenance):
    pack=first.prepare(data,tr,ev)
    # Gene standardization was fitted to unique TRAINING in vitro source conditions.
    gene_expression=pack['source'].numpy()
    scores=(gene_expression.astype(np.float64)@projection).astype(np.float32)
    unique_sources=np.unique(data.source_ids[tr])
    pathway_scale=StandardScaler().fit(scores[unique_sources])
    normalized=pathway_scale.transform(scores).astype(np.float32)
    pack['source']=torch.cat([pack['source'],torch.tensor(normalized)],dim=1)
    pack['preprocess'].update(pathway_scale=pathway_scale,pathway_projection=projection,pathway_ids=pathways.pathway.tolist(),pathway_provenance=provenance)
    pack['gene_dim']=data.source.shape[1];pack['pathway_dim']=projection.shape[1]
    return pack


class PathwayStructureMLP(nn.Module):
    def __init__(self,gene_dim,pathway_dim,condition_dim,structure_dim,output_dim,use_pathway):
        super().__init__();self.gene_dim=gene_dim;self.use_pathway=use_pathway
        self.expression_encoder=nn.Sequential(nn.Linear(gene_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU())
        self.structure_encoder=nn.Sequential(nn.Linear(structure_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,32,bias=False),nn.SiLU())
        self.pathway_encoder=nn.Sequential(nn.Linear(pathway_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,32,bias=False),nn.SiLU())
        self.decoder=nn.Sequential(nn.Linear(64+32+32+condition_dim,64),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,output_dim))

    def forward(self,source,condition,structure):
        genes=source[:,:self.gene_dim];pathways=source[:,self.gene_dim:]
        if not self.use_pathway:pathways=torch.zeros_like(pathways)
        return self.decoder(torch.cat([self.expression_encoder(genes),self.structure_encoder(structure),self.pathway_encoder(pathways),condition],dim=1))


def run_pathway_comparison(a):
    projection,pathways,provenance=load_projection(a.annotation,a.prepared)
    a.out.mkdir(parents=True,exist_ok=True)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(projection_sha256=hashlib.sha256(np.ascontiguousarray(projection).tobytes()).hexdigest(),protocol='matched four-branch MLP ablation; structure/pathway slots zeroed when absent',pathway_score='gene-weighted mean of training-standardized in vitro expression, then training-fitted pathway standardization',architecture='expression128_64_structure64_32_pathway64_32_fusion64_fullprobe')
    config_path=a.out/'config.json'
    if config_path.exists():
        if json.loads(config_path.read_text())!=config:raise ValueError('Resume configuration mismatch')
    else:
        if any(a.out.iterdir()):raise ValueError('Unrecognized output directory')
        config_path.write_text(json.dumps(config,indent=2))
    progress_path=a.out/'progress.json'
    progress=json.loads(progress_path.read_text()) if progress_path.is_file() else {'groups':{}}
    def save_progress():
        temporary=progress_path.with_suffix('.tmp');temporary.write_text(json.dumps(progress,indent=2));temporary.replace(progress_path)
    pathways.to_csv(a.out/'pathway_audit.csv',index=False)
    (a.out/'pathway_provenance.json').write_text(json.dumps(provenance,ensure_ascii=False,indent=2))
    print(f'Pathways={len(pathways)}; probes in retained pathways={int(np.any(projection>0,axis=1).sum())}',flush=True)
    tables=[];split_reference=None
    for enabled in [False,True]:
        key='with_pathway' if enabled else 'without_pathway';entry=progress['groups'].get(key)
        if entry and (Path(entry['path'])/'comparison.csv').is_file():
            folder=Path(entry['path']);result=pd.read_csv(folder/'comparison.csv')
            print(key,': completed group reused',flush=True)
        else:
            if entry:print(key,': partial folder preserved; this two-model group restarts',flush=True)
            folder=a.out/f'{key}_{uuid.uuid4().hex[:6]}'
            progress['groups'][key]={'path':str(folder),'status':'running'};save_progress()
            values={**vars(a),'out':folder,'pathway_enabled':enabled,'architecture':config['architecture']}
            group_args=SimpleNamespace(**values)
            def prepare_fn(data,tr,ev):return prepare_pathways(data,tr,ev,projection,pathways,provenance)
            def factory(data,pack,seed):
                first.seed_all(seed)
                return PathwayStructureMLP(pack['gene_dim'],pack['pathway_dim'],pack['conditions'].shape[1],pack['structure'].shape[1],data.target.shape[1],enabled)
            print(f'\nStarting {key}: structure off/on, output={folder}',flush=True)
            result=first.run(group_args,model_factory=factory,prepare_fn=prepare_fn)
        if set(result.model)!={'expression_only','expression_structure'}:raise ValueError('Incomplete model group')
        split=pd.read_csv(folder/'compound_split.csv').sort_values('compound').reset_index(drop=True)
        if split_reference is None:split_reference=split
        else:pd.testing.assert_frame_equal(split_reference,split)
        group_cfg=json.loads((folder/'config.json').read_text())
        if group_cfg.get('pathway_enabled')!=enabled:raise ValueError('Cached pathway flag mismatch')
        for field in ['seed','epochs','check_every','batch_size']:
            if group_cfg[field]!=getattr(a,field):raise ValueError('Cached group configuration mismatch')
        result['pathway']=enabled;result['structure']=result.model.eq('expression_structure')
        result['model']=['expression+structure+pathway' if enabled and s else 'expression+pathway' if enabled else 'expression+structure' if s else 'expression' for s in result.structure]
        tables.append(result)
        progress['groups'][key]['status']='complete';save_progress()
        pd.concat(tables,ignore_index=True).to_csv(a.out/'comparison.csv',index=False)
    result=pd.concat(tables,ignore_index=True)
    if result.parameters.nunique()!=1:raise ValueError('Unmatched parameter counts')
    keys=['n_target_conditions','n_treated_conditions','zero_response_rmse']
    for key in keys:
        if not np.allclose(result[key],result[key].iloc[0]):raise ValueError('Evaluation targets/baseline mismatch')
    result.to_csv(a.out/'comparison.csv',index=False)
    print(result.to_string(index=False));print('Saved:',a.out,flush=True)
    return result



## 원본 86번째 셀 · In[93]


In [93]:
from datetime import datetime
import uuid

DATA_ROOT = Path('/home/seowon1211/data1')
ANNOTATION = DATA_ROOT/'conditioned_expression/kegg_rat_20261102_034010_265aed'
EPOCHS = 30

# 새 커널에서 재개할 때만 이전 결과 폴더 지정
RESUME_ROOT = None
if RESUME_ROOT is None:
    PATHWAY_RUN = DATA_ROOT/'conditioned_expression'/(
        'pathway_structure_mlp_' + datetime.now().strftime('%Y%m%d_%H%M%S')
        + '_' + uuid.uuid4().hex[:6]
    )
else:
    PATHWAY_RUN = Path(RESUME_ROOT)

a_pathway = SimpleNamespace(
    prepared=DATA_ROOT/'conditioned_expression/prepared_controls',
    structure=DATA_ROOT/'conditioned_expression/morgan_matched/morgan_binary.csv',
    annotation=ANNOTATION,out=PATHWAY_RUN,
    epochs=EPOCHS,check_every=5,batch_size=128,threads=4,seed=42,
)
print('KEGG snapshot:',ANNOTATION,flush=True)
print('새 결과/재개 폴더:',PATHWAY_RUN,flush=True)


KEGG snapshot: /home/seowon1211/data1/conditioned_expression/kegg_rat_20261102_034010_265aed
새 결과/재개 폴더: /home/seowon1211/data1/conditioned_expression/pathway_structure_mlp_20261102_035406_327bbc


## 원본 87번째 셀 · In[94]


In [94]:
print('MLP 네 조건 비교 시작',flush=True)
with threadpool_limits(limits=a_pathway.threads):
    pathway_comparison=run_pathway_comparison(a_pathway)
print('네 조건 비교 완료',flush=True)


MLP 네 조건 비교 시작
Pathways=321; probes in retained pathways=2358

Starting without_pathway: structure off/on, output=/home/seowon1211/data1/conditioned_expression/pathway_structure_mlp_20261102_035406_327bbc/without_pathway_04d011
Compounds train/val/test: [89, 23, 28]; probes=3453
Missing structure compounds: ['LPS', 'TNFÉø', 'tunicamycin']
expression_only epoch=1/30, val_RMSE=0.28800, best_epoch=1
expression_only epoch=5/30, val_RMSE=0.28751, best_epoch=5
expression_only epoch=10/30, val_RMSE=0.28140, best_epoch=10
expression_only epoch=15/30, val_RMSE=0.28315, best_epoch=10
expression_only epoch=20/30, val_RMSE=0.28704, best_epoch=10
expression_only epoch=25/30, val_RMSE=0.29245, best_epoch=10
expression_only epoch=30/30, val_RMSE=0.30043, best_epoch=10
expression_structure epoch=1/30, val_RMSE=0.29452, best_epoch=1
expression_structure epoch=5/30, val_RMSE=0.29604, best_epoch=1
expression_structure epoch=10/30, val_RMSE=0.29244, best_epoch=10
expression_structure epoch=15/30, val_RMSE

## 원본 88번째 셀 · In[95]


In [95]:
print('첫 미팅용 4행 비교표')
display(pathway_comparison[['model','structure','pathway','absolute_rmse',
    'response_rmse','zero_response_rmse','ratio_to_no_change','pair_rmse',
    'selected_epoch','parameters']])

indexed=pathway_comparison.set_index(['structure','pathway'])
effects=[]
for label,baseline,augmented in [
    ('구조만 추가',(False,False),(True,False)),
    ('pathway만 추가',(False,False),(False,True)),
    ('구조가 있을 때 pathway 추가',(True,False),(True,True)),
    ('pathway가 있을 때 구조 추가',(False,True),(True,True)),
]:
    before=indexed.loc[baseline];after=indexed.loc[augmented]
    effects.append(dict(comparison=label,
        absolute_RMSE_change=after.absolute_rmse-before.absolute_rmse,
        response_RMSE_change=after.response_rmse-before.response_rmse))
effect_table=pd.DataFrame(effects)
print('음수면 추가 모델의 오차가 작습니다. 두 지표를 함께 보세요.')
display(effect_table)
effect_table.to_csv(PATHWAY_RUN/'input_effects.csv',index=False)
print('결과:',PATHWAY_RUN/'comparison.csv')


첫 미팅용 4행 비교표


,model,structure,pathway,absolute_rmse,response_rmse,zero_response_rmse,ratio_to_no_change,pair_rmse,selected_epoch,parameters
0,expression,False,False,0.324497,0.312818,0.330116,0.947600,0.323218,10,839933
1,expression+structure,True,False,0.338225,0.311451,0.330116,0.943459,0.333547,10,839933
2,expression+pathway,False,True,0.324664,0.316732,0.330116,0.959456,0.323025,15,839933
3,expression+structure+pathway,True,True,0.333340,0.309828,0.330116,0.938541,0.329823,10,839933


음수면 추가 모델의 오차가 작습니다. 두 지표를 함께 보세요.


,comparison,absolute_RMSE_change,response_RMSE_change
0,구조만 추가,0.013727,-0.001367
1,pathway만 추가,0.000167,0.003914
2,구조가 있을 때 pathway 추가,-0.004884,-0.001624
3,pathway가 있을 때 구조 추가,0.008676,-0.006904


결과: /home/seowon1211/data1/conditioned_expression/pathway_structure_mlp_20261102_035406_327bbc/comparison.csv


## 원본 89번째 셀 · In[96]


In [96]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a,model_factory=None,prepare_fn=None):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    model_factory=model_for if model_factory is None else model_factory
    prepare_fn=prepare if prepare_fn is None else prepare_fn
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare_fn(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in [('expression_only',False),('expression_structure',True)]:
        model=model_factory(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare_fn(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_factory(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



## 원본 90번째 셀 · In[97]


In [97]:
from types import SimpleNamespace
first = SimpleNamespace(Data=Data,prepare=prepare,inputs=inputs,evaluate=evaluate,seed_all=seed_all)
def load_projection(annotation,prepared):
    annotation=Path(annotation)
    provenance=json.loads((annotation/'provenance.json').read_text())
    probes=pd.read_csv(Path(prepared)/'probes.csv')['probe'].astype(str).tolist()
    if provenance['species']!='Rattus norvegicus' or provenance['probe_order']!=probes:raise ValueError('Species/probe order mismatch')
    projection=np.load(annotation/'projection.npy').astype(np.float64)
    pathways=pd.read_csv(annotation/'pathway_audit.csv')
    if projection.shape!=(len(probes),len(pathways)) or len(pathways)==0 or pathways.pathway.duplicated().any():raise ValueError('Pathway layout mismatch')
    if not np.isfinite(projection).all() or (projection<0).any():raise ValueError('Invalid pathway weights')
    np.testing.assert_allclose(projection.sum(0),1.,rtol=0,atol=1e-6)
    return projection,pathways,provenance

def prepare_pathways(data,tr,ev,projection,pathways,provenance):
    pack=first.prepare(data,tr,ev)
    # Gene standardization was fitted to unique TRAINING in vitro source conditions.
    gene_expression=pack['source'].numpy()
    scores=(gene_expression.astype(np.float64)@projection).astype(np.float32)
    unique_sources=np.unique(data.source_ids[tr])
    pathway_scale=StandardScaler().fit(scores[unique_sources])
    normalized=pathway_scale.transform(scores).astype(np.float32)
    pack['source']=torch.cat([pack['source'],torch.tensor(normalized)],dim=1)
    pack['preprocess'].update(pathway_scale=pathway_scale,pathway_projection=projection,pathway_ids=pathways.pathway.tolist(),pathway_provenance=provenance)
    pack['gene_dim']=data.source.shape[1];pack['pathway_dim']=projection.shape[1]
    return pack

class PathwayStructureMLP(nn.Module):
    def __init__(self,gene_dim,pathway_dim,condition_dim,structure_dim,output_dim,use_pathway):
        super().__init__();self.gene_dim=gene_dim;self.use_pathway=use_pathway
        self.expression_encoder=nn.Sequential(nn.Linear(gene_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU())
        self.structure_encoder=nn.Sequential(nn.Linear(structure_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,32,bias=False),nn.SiLU())
        self.pathway_encoder=nn.Sequential(nn.Linear(pathway_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,32,bias=False),nn.SiLU())
        self.decoder=nn.Sequential(nn.Linear(64+32+32+condition_dim,64),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,output_dim))

    def forward(self,source,condition,structure):
        genes=source[:,:self.gene_dim];pathways=source[:,self.gene_dim:]
        if not self.use_pathway:pathways=torch.zeros_like(pathways)
        return self.decoder(torch.cat([self.expression_encoder(genes),self.structure_encoder(structure),self.pathway_encoder(pathways),condition],dim=1))
pathway = SimpleNamespace(load_projection=load_projection,prepare_pathways=prepare_pathways,PathwayStructureMLP=PathwayStructureMLP)


## 원본 91번째 셀 · In[98]


In [98]:
"""Four input conditions across MLP references, pure cGAN, and epsilon-DDPM."""
import argparse
import copy
import hashlib
import json
from pathlib import Path
from types import SimpleNamespace
import uuid
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from threadpoolctl import threadpool_limits




class Context(nn.Module):
    def __init__(self,dims,use_pathway):
        super().__init__();gene,path,cond,structure,out=dims
        template=pathway.PathwayStructureMLP(gene,path,cond,structure,out,use_pathway)
        self.gene_dim=gene;self.use_pathway=use_pathway
        self.expression_encoder=template.expression_encoder;self.structure_encoder=template.structure_encoder;self.pathway_encoder=template.pathway_encoder
        self.fusion=template.decoder[:3]
    def forward(self,source,condition,structure):
        genes=source[:,:self.gene_dim];scores=source[:,self.gene_dim:]
        if not self.use_pathway:scores=torch.zeros_like(scores)
        return self.fusion(torch.cat([self.expression_encoder(genes),self.structure_encoder(structure),self.pathway_encoder(scores),condition],1))


class Generator(nn.Module):
    def __init__(self,dims,use_pathway):
        super().__init__();self.context=Context(dims,use_pathway)
        self.head=nn.Sequential(nn.Linear(64+16,64),nn.SiLU(),nn.Linear(64,dims[-1]))
    def decode(self,context,noise):return self.head(torch.cat([context,noise],1))
    def forward(self,*args):return self.decode(self.context(*args[:3]),args[3])


class Discriminator(nn.Module):
    def __init__(self,dims,use_pathway):
        super().__init__();self.context=Context(dims,use_pathway)
        self.head=nn.Sequential(nn.Linear(64+dims[-1],128),nn.LeakyReLU(.2),nn.Linear(128,64),nn.LeakyReLU(.2),nn.Linear(64,1))
    def forward(self,source,condition,structure,expression):return self.head(torch.cat([self.context(source,condition,structure),expression],1)).squeeze(1)


class EpsilonDenoiser(nn.Module):
    def __init__(self,dims,use_pathway,steps):
        super().__init__();self.context=Context(dims,use_pathway);self.time=nn.Embedding(steps,16);self.output_dim=dims[-1]
        self.correction=nn.Sequential(nn.Linear(64+self.output_dim+16,128),nn.SiLU(),nn.Linear(128,128),nn.SiLU(),nn.Linear(128,self.output_dim))
        nn.init.zeros_(self.correction[-1].weight);nn.init.zeros_(self.correction[-1].bias)
    def decode(self,context,noisy,t):
        # A neural residual connection, not a separate prediction/post-correction model.
        return noisy+self.correction(torch.cat([context,noisy,self.time(t)],1))
    def forward(self,source,condition,structure,noisy,t):return self.decode(self.context(source,condition,structure),noisy,t)


def ddpm_schedule(steps):
    # Standard linear DDPM beta schedule rescaled to the chosen time discretization.
    beta=torch.linspace(1e-4,.02,steps)*(1000/steps)
    beta=beta.clamp(max=.999);alpha=1-beta
    return beta,alpha,torch.cumprod(alpha,0)


def make_models(kind,data,pack,use_pathway,a):
    first.seed_all(a.seed)
    dims=(pack['gene_dim'],pack['pathway_dim'],pack['conditions'].shape[1],pack['structure'].shape[1],data.target.shape[1])
    model=Generator(dims,use_pathway) if kind=='cgan' else EpsilonDenoiser(dims,use_pathway,a.ddpm_steps)
    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
    disc=Discriminator(dims,use_pathway) if kind=='cgan' else None
    dopt=torch.optim.AdamW(disc.parameters(),lr=.001,weight_decay=.01) if disc is not None else None
    return model,opt,disc,dopt


def train_epoch(kind,model,opt,disc,dopt,data,pack,tr,structure,a):
    model.train()
    if disc is not None:disc.train()
    total=0.;_,_,abar=ddpm_schedule(a.ddpm_steps)
    for local in torch.randperm(len(tr)).split(a.batch_size):
        rows=tr[local.numpy()];inputs=first.inputs(data,pack,rows,structure)
        truth=pack['target'][data.target_ids[rows]];weights=pack['weights'][local]
        if kind=='cgan':
            for parameter in disc.parameters():parameter.requires_grad_(True)
            fake=model(*inputs,torch.randn(len(rows),16)).detach()
            real_logits=disc(*inputs,truth);fake_logits=disc(*inputs,fake)
            loss_d=.5*((F.binary_cross_entropy_with_logits(real_logits,torch.ones_like(real_logits),reduction='none')+F.binary_cross_entropy_with_logits(fake_logits,torch.zeros_like(fake_logits),reduction='none'))*weights).mean()
            if not torch.isfinite(loss_d):raise ValueError('Nonfinite discriminator loss')
            dopt.zero_grad();loss_d.backward();nn.utils.clip_grad_norm_(disc.parameters(),1.);dopt.step()
            for parameter in disc.parameters():parameter.requires_grad_(False)
            pred=model(*inputs,torch.randn(len(rows),16));logits=disc(*inputs,pred)
            # Pure conditional GAN: no reconstruction, cycle, module, or pathway loss.
            loss=(F.binary_cross_entropy_with_logits(logits,torch.ones_like(logits),reduction='none')*weights).mean()
        else:
            t=torch.randint(a.ddpm_steps,(len(rows),));noise=torch.randn_like(truth)
            noisy=abar[t,None].sqrt()*truth+(1-abar[t,None]).sqrt()*noise
            epsilon=model(*inputs,noisy,t)
            loss=((epsilon-noise).square().mean(1)*weights).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step();total+=float(loss.detach())*len(rows)
    return total/len(tr)


@torch.no_grad()
def draw_mean(kind,model,inputs,a,seed):
    model.eval();gen=torch.Generator().manual_seed(seed)
    context=model.context(*inputs);n=len(context);context=context.repeat_interleave(a.samples,0)
    if kind=='cgan':out=model.decode(context,torch.randn((len(context),16),generator=gen))
    else:
        beta,alpha,abar=ddpm_schedule(a.ddpm_steps)
        out=torch.randn((len(context),model.output_dim),generator=gen)
        for step in reversed(range(a.ddpm_steps)):
            epsilon=model.decode(context,out,torch.full((len(context),),step,dtype=torch.long))
            mean=(out-beta[step]/(1-abar[step]).sqrt()*epsilon)/alpha[step].sqrt()
            out=mean
            if step:
                posterior_variance=beta[step]*(1-abar[step-1])/(1-abar[step])
                out=out+posterior_variance.sqrt()*torch.randn(out.shape,generator=gen)
    if not torch.isfinite(out).all():raise ValueError('Nonfinite DDPM/GAN samples')
    return out.reshape(n,a.samples,-1).mean(1).numpy()


@torch.no_grad()
def predict(kind,model,data,pack,rows,structure,a,seed):
    ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),float);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),a.batch_size):
        batch=rows[start:start+a.batch_size]
        pred=draw_mean(kind,model,first.inputs(data,pack,batch,structure),a,seed+start)+pack['target_mean']
        targets=data.target_ids[batch];indices=np.array([lookup[int(cid)] for cid in targets])
        np.add.at(sums,indices,pred);np.add.at(counts,indices,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[targets])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def save_json(path,value):
    temp=path.with_suffix('.tmp');temp.write_text(json.dumps(value,indent=2));temp.replace(path)


def load_mlp_reference(a,data,projection):
    root=Path(a.mlp_reference);cfg=json.loads((root/'config.json').read_text())
    for key in ['prepared','structure','annotation']:
        if Path(cfg[key]).resolve()!=Path(getattr(a,key)).resolve():raise ValueError(f'MLP reference {key} mismatch')
    for key in ['seed','epochs','check_every','batch_size']:
        if cfg[key]!=getattr(a,key):raise ValueError(f'MLP reference {key} mismatch')
    if cfg.get('projection_sha256')!=hashlib.sha256(np.ascontiguousarray(projection).tobytes()).hexdigest():raise ValueError('Pathway snapshot changed')
    summary=pd.read_csv(root/'comparison.csv')
    if len(summary)!=4 or len(summary[['structure','pathway']].drop_duplicates())!=4:raise ValueError('Four completed MLP conditions required')
    progress=json.loads((root/'progress.json').read_text());reference_split=None
    for pathflag in [False,True]:
        key='with_pathway' if pathflag else 'without_pathway';folder=Path(progress['groups'][key]['path'])
        split=pd.read_csv(folder/'compound_split.csv').sort_values('compound').reset_index(drop=True)
        if reference_split is None:reference_split=split
        else:pd.testing.assert_frame_equal(split,reference_split)
        for structure,name in [(False,'expression_only'),(True,'expression_structure')]:
            with np.load(folder/f'{name}_predictions.npz') as p:ids,pred=p['target_ids'],p['prediction']
            metrics,_=first.evaluate(data,ids,pred)
            record=summary.loc[summary.structure.eq(structure)&summary.pathway.eq(pathflag)].iloc[0]
            for metric in ['absolute_rmse','response_rmse']:np.testing.assert_allclose(metrics[metric],record[metric],rtol=1e-5,atol=1e-6)
    summary=summary.copy();summary['family']='MLP';summary['origin']='reused_MLP_reference';summary['discriminator_parameters']=0
    return summary,reference_split


def run_matrix(a,families=('cgan','ddpm')):
    if not set(families)<={'cgan','ddpm'}:raise ValueError('Select cgan and/or ddpm')
    projection,pathways,provenance=pathway.load_projection(a.annotation,a.prepared)
    torch.set_num_threads(a.threads);data=first.Data(a)
    mlp,split=load_mlp_reference(a,data,projection)
    if split.compound.duplicated().any() or set(split.compound)!=set(data.names):raise ValueError('Compound split mismatch')
    def rows(stage):return np.flatnonzero(data.pairs.compound.isin(set(split.loc[split.stage.eq(stage),'compound'])))
    itr,iv,te=rows('inner_train'),rows('inner_validation'),rows('outer_test');tr=np.sort(np.r_[itr,iv])
    if any(set(x)&set(y) for x,y in [(itr,iv),(itr,te),(iv,te)]):raise ValueError('Split overlap')
    a.out.mkdir(parents=True,exist_ok=True)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(gan='conditional BCE only; no reconstruction loss',diffusion='epsilon-DDPM; scaled linear betas; all reverse steps; no DDIM/v/clipping',denoiser='residual MLP epsilon predictor with identity skip, not post-correction',samples_mean='mean over generated samples per pair, then mean over sources per target')
    config_path=a.out/'config.json'
    if config_path.exists():
        if json.loads(config_path.read_text())!=config:raise ValueError('Resume settings mismatch')
    else:
        if any(a.out.iterdir()):raise ValueError('Unrecognized result directory')
        save_json(config_path,config)
    progress_path=a.out/'progress.json';progress=json.loads(progress_path.read_text()) if progress_path.is_file() else {'runs':{}}
    split.to_csv(a.out/'compound_split.csv',index=False)
    inner=pathway.prepare_pathways(data,itr,iv,projection,pathways,provenance)
    outer=pathway.prepare_pathways(data,tr,te,projection,pathways,provenance)
    joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    def collect():
        frames=[mlp]
        for entry in progress['runs'].values():
            path=Path(entry['path'])/'metrics.json'
            if path.is_file():frames.append(pd.DataFrame([json.loads(path.read_text())]))
        table=pd.concat(frames,ignore_index=True);table.to_csv(a.out/'comparison.csv',index=False);return table
    print('Four MLP references verified; no MLP retraining.',flush=True)
    print(f'New families={list(families)}; DDPM full reverse steps={a.ddpm_steps}; generated samples/pair={a.samples}',flush=True)
    collect()
    for kind in families:
        for structure in [False,True]:
            for pathflag in [False,True]:
                name=f'{kind}_structure{int(structure)}_pathway{int(pathflag)}';entry=progress['runs'].get(name)
                if entry and (Path(entry['path'])/'metrics.json').is_file():print(name,': complete; skipped',flush=True);continue
                if entry:print(name,': partial attempt retained; restarting this variant',flush=True)
                folder=a.out/f'{name}_{uuid.uuid4().hex[:6]}';folder.mkdir()
                progress['runs'][name]={'path':str(folder),'status':'running'};save_json(progress_path,progress)
                print(f'\n{name}: training starts; {folder}',flush=True)
                model,opt,disc,dopt=make_models(kind,data,inner,pathflag,a)
                best=float('inf');epoch_best=None;state=None;history=[]
                for epoch in range(1,a.epochs+1):
                    loss=train_epoch(kind,model,opt,disc,dopt,data,inner,itr,structure,a)
                    if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                        ids,pred,_=predict(kind,model,data,inner,iv,structure,a,a.seed+20000)
                        mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                        if mse<best:best=mse;epoch_best=epoch;state=copy.deepcopy(model.state_dict())
                        history.append(dict(epoch=epoch,loss=loss,validation_rmse=np.sqrt(mse),best_epoch=epoch_best))
                        pd.DataFrame(history).to_csv(folder/'validation_history.csv',index=False)
                        print(f'{name} epoch={epoch}/{a.epochs} val_RMSE={np.sqrt(mse):.5f} best_epoch={epoch_best}',flush=True)
                torch.save(state,folder/'inner_best.pt')
                model,opt,disc,dopt=make_models(kind,data,outer,pathflag,a)
                for epoch in range(1,epoch_best+1):
                    train_epoch(kind,model,opt,disc,dopt,data,outer,tr,structure,a)
                    if epoch==1 or epoch%10==0 or epoch==epoch_best:print(f'{name} refit={epoch}/{epoch_best}',flush=True)
                ids,pred,pair_rmse=predict(kind,model,data,outer,te,structure,a,a.seed+30000)
                metrics,detail=first.evaluate(data,ids,pred)
                metrics.update(model=name,family='cGAN' if kind=='cgan' else 'epsilon-DDPM',structure=structure,pathway=pathflag,origin='new_vanilla_run',selected_epoch=epoch_best,validation_rmse=float(np.sqrt(best)),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()),discriminator_parameters=sum(p.numel() for p in disc.parameters()) if disc is not None else 0)
                torch.save(model.state_dict(),folder/'model.pt')
                if disc is not None:torch.save(disc.state_dict(),folder/'discriminator.pt')
                np.savez_compressed(folder/'predictions.npz',target_ids=ids,prediction=pred)
                detail.to_csv(folder/'response_condition_metrics.csv',index=False)
                save_json(folder/'metrics.json',metrics);progress['runs'][name]['status']='complete';save_json(progress_path,progress)
                collect();print(f'{name}: TEST response RMSE={metrics["response_rmse"]:.5f}',flush=True)
    result=collect();print(result.to_string(index=False));print('Saved:',a.out,flush=True);return result



## 원본 92번째 셀 · In[99]


In [99]:
from datetime import datetime
import uuid

DATA_ROOT=Path('/home/seowon1211/data1')
MLP_REFERENCE=DATA_ROOT/'conditioned_expression/pathway_structure_mlp_20261102_035406_327bbc'
assert (MLP_REFERENCE/'comparison.csv').is_file(),MLP_REFERENCE
reference_config=json.loads((MLP_REFERENCE/'config.json').read_text())

RESUME_ROOT=None
if RESUME_ROOT is None:
    MATRIX_OUT=DATA_ROOT/'conditioned_expression'/(
        'vanilla_information_matrix_'+datetime.now().strftime('%Y%m%d_%H%M%S')
        +'_'+uuid.uuid4().hex[:6]
    )
else:
    MATRIX_OUT=Path(RESUME_ROOT)

a_matrix=SimpleNamespace(
    prepared=Path(reference_config['prepared']),structure=Path(reference_config['structure']),
    annotation=Path(reference_config['annotation']),mlp_reference=MLP_REFERENCE,out=MATRIX_OUT,
    epochs=reference_config['epochs'],check_every=reference_config['check_every'],
    batch_size=reference_config['batch_size'],threads=4,seed=reference_config['seed'],
    samples=4,ddpm_steps=100,
)
FAMILIES=['cgan','ddpm']
print('MLP4조건 참조:',MLP_REFERENCE,flush=True)
print('결과/재개 폴더:',MATRIX_OUT,flush=True)


MLP4조건 참조: /home/seowon1211/data1/conditioned_expression/pathway_structure_mlp_20261102_035406_327bbc
결과/재개 폴더: /home/seowon1211/data1/conditioned_expression/vanilla_information_matrix_20261102_040702_9210e0


## 원본 93번째 셀 · In[100]


In [100]:
print('기본 모델 12조건 비교 시작: MLP 재사용, cGAN/DDPM 추가',flush=True)
with threadpool_limits(limits=a_matrix.threads):
    matrix_results=run_matrix(a_matrix,FAMILIES)
print('선택한 계열 실행 완료',flush=True)


기본 모델 12조건 비교 시작: MLP 재사용, cGAN/DDPM 추가
Four MLP references verified; no MLP retraining.
New families=['cgan', 'ddpm']; DDPM full reverse steps=100; generated samples/pair=4

cgan_structure0_pathway0: training starts; /home/seowon1211/data1/conditioned_expression/vanilla_information_matrix_20261102_040702_9210e0/cgan_structure0_pathway0_62a53d
cgan_structure0_pathway0 epoch=1/30 val_RMSE=0.37637 best_epoch=1
cgan_structure0_pathway0 epoch=5/30 val_RMSE=0.34727 best_epoch=5
cgan_structure0_pathway0 epoch=10/30 val_RMSE=0.33503 best_epoch=10
cgan_structure0_pathway0 epoch=15/30 val_RMSE=0.33335 best_epoch=15
cgan_structure0_pathway0 epoch=20/30 val_RMSE=0.33750 best_epoch=15
cgan_structure0_pathway0 epoch=25/30 val_RMSE=0.32754 best_epoch=25
cgan_structure0_pathway0 epoch=30/30 val_RMSE=0.36038 best_epoch=25
cgan_structure0_pathway0 refit=1/25
cgan_structure0_pathway0 refit=10/25
cgan_structure0_pathway0 refit=20/25
cgan_structure0_pathway0 refit=25/25
cgan_structure0_pathway0: TEST resp

## 원본 94번째 셀 · In[101]


In [101]:
print('현재 완료된 조건 — 모두 완료하면12행입니다.')
display(matrix_results[['family','structure','pathway','absolute_rmse','response_rmse',
    'zero_response_rmse','ratio_to_no_change','pair_rmse','selected_epoch','parameters','discriminator_parameters']])
print('반응 RMSE 비교표')
display(matrix_results.pivot(index=['structure','pathway'],columns='family',values='response_rmse'))
print('절대 발현 RMSE 비교표')
display(matrix_results.pivot(index=['structure','pathway'],columns='family',values='absolute_rmse'))
print('결과:',MATRIX_OUT/'comparison.csv')


현재 완료된 조건 — 모두 완료하면12행입니다.


,family,structure,pathway,absolute_rmse,response_rmse,zero_response_rmse,ratio_to_no_change,pair_rmse,selected_epoch,parameters,discriminator_parameters
0,MLP,False,False,0.324497,0.312818,0.330116,0.947600,0.323218,10,839933,0
1,MLP,True,False,0.338225,0.311451,0.330116,0.943459,0.333547,10,839933,0
2,MLP,False,True,0.324664,0.316732,0.330116,0.959456,0.323025,15,839933,0
3,MLP,True,True,0.333340,0.309828,0.330116,0.938541,0.329823,10,839933,0
4,cGAN,False,False,0.385143,0.333167,0.330116,1.009240,0.389184,25,845117,1074113
5,cGAN,False,True,0.375271,0.334028,0.330116,1.011851,0.380531,30,845117,1074113
6,cGAN,True,False,0.398807,0.332429,0.330116,1.007006,0.399271,10,845117,1074113
7,cGAN,True,True,0.374362,0.333284,0.330116,1.009594,0.380863,25,845117,1074113
8,epsilon-DDPM,False,False,0.333844,0.332755,0.330116,1.007993,0.396741,25,1531389,0
9,epsilon-DDPM,False,True,0.333497,0.329794,0.330116,0.999022,0.396786,25,1531389,0


반응 RMSE 비교표


family                  MLP      cGAN  epsilon-DDPM
structure pathway                                  
False     False    0.312818  0.333167      0.332755
          True     0.316732  0.334028      0.329794
True      False    0.311451  0.332429      0.331319
          True     0.309828  0.333284      0.334520

절대 발현 RMSE 비교표


family                  MLP      cGAN  epsilon-DDPM
structure pathway                                  
False     False    0.324497  0.385143      0.333844
          True     0.324664  0.375271      0.333497
True      False    0.338225  0.398807      0.359125
          True     0.333340  0.374362      0.360968

결과: /home/seowon1211/data1/conditioned_expression/vanilla_information_matrix_20261102_040702_9210e0/comparison.csv
